# Sprint 5

## Install PySpark

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

## Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Import the funtions and create data path

In [ ]:
from pathlib import Path

from pyspark.sql.window import Window
import pyspark.sql.functions as F 
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    avg,
    count,
    count_distinct,
    broadcast,
    substring,
    min as spark_min,
    max as spark_max,
    mean,
    approx_percentile
)

DATA_DIR = Path("../data/MIMIC-IV/hosp")
GENERAL_DATA_DIR = Path("../data")
EVIDENCE_DIR = Path("../out/evidence")
VISIT_DF = Path("../data/visits")
DIAGNOSES_DF = Path("../data/diagnoses")
ICD_CODES_DF = Path("../data/icd_codes")

## Dataframes

In [ ]:
# -------------------------------------------------------
# Pulling Dataframes from Saved Parquet Files
# -------------------------------------------------------
visits = spark.read.parquet(str(VISIT_DF))
visits.show(10, truncate=False)

diagnoses = spark.read.parquet(str(DIAGNOSES_DF))
diagnoses.show(10, truncate=False)

icd_codes = spark.read.parquet(str(ICD_CODES_DF))
icd_codes.show(10, truncate=False)



In [ ]:
# Finding out the top Relevant symptoms from each
# Making sure to have one output that merges

#defining a window for the top ranked row to only use that per hadm_id

print( diagnoses.groupBy("subject_id"))

top_symptoms = (
    diagnoses
    .filter( col("visit_type")=="SYMPTOM")
    .join(
        broadcast(icd_codes.filter(col("status") == "RELEVANT")), #broadcast join on filtered dataset
        on=["icd_code","icd_version"],
        how="inner"
    )
    .where(col("ranking") <= 10)
    .withColumn("cons_icd_code",
         when(       )

    )
    .groupBy("icd_code") # decided to just go for top 10, will figure out
    .agg(
        count("*").alias("total_count"),
        count_distinct("subject_id").alias("subject_count")
)
    .sort("total_count", ascending=False)
)

top_symptoms.show(10, truncate=False)


In [ ]:
# Visit Frequency Relative to Diagnosis

import matplotlib.pyplot as plt
from pyspark.sql.functions import when, datediff

# Get BC first diagnosis date per patient
bc_dx_dates = (
    visits
    .filter(col("visit_type") == "BC_FIRST_DX")
    .select(
        col("subject_id"),
        col("admit_day").alias("dx_date")
    )
)

# Join symptom visits to their patient's dx date, compute days_before_dx
visits_with_days = (
    visits
    .filter(col("visit_type") == "SYMPTOM")
    .join(broadcast(bc_dx_dates), on="subject_id", how="inner")
    .withColumn("days_before_dx", datediff(col("dx_date"), col("admit_day")))
    .filter(col("days_before_dx") >= 0)
)

# Bucket into time groups
visits_bucketed = (
    visits_with_days
    .withColumn(
        "time_bucket",
        when(col("days_before_dx") <= 30,  "0-30 days")
        .when(col("days_before_dx") <= 90,  "30-90 days")
        .when(col("days_before_dx") <= 180, "90-180 days")
        .otherwise("180+ days")
    )
)

# Aggregate — chronological order
bucket_order = (
    when(col("time_bucket") == "180+ days",   0)
    .when(col("time_bucket") == "90-180 days", 1)
    .when(col("time_bucket") == "30-90 days",  2)
    .otherwise(3)
)

visit_frequency_relative_dx = (
    visits_bucketed
    .groupBy("time_bucket")
    .agg(
        count("hadm_id").alias("visit_count"),
        count_distinct("subject_id").alias("distinct_patients"),
        (count("hadm_id") / count_distinct("subject_id")).alias("avg_visits_per_patient")
    )
    .orderBy(bucket_order)
)

print("=== visit_frequency_relative_dx ===")
visit_frequency_relative_dx.show(truncate=False)

# Bar chart
# matplotlib cannot read Spark dataframes directly without converting to pandas
pdf = visit_frequency_relative_dx.toPandas()


fig_visit_frequency_relative_dx, ax = plt.subplots()
ax.bar(pdf["time_bucket"], pdf["avg_visits_per_patient"])
ax.set_xlabel("Time Bucket (Days Before Diagnosis)")
ax.set_ylabel("Avg Visits per Patient")
ax.set_title("Hospital Visit Frequency Relative to BC Diagnosis")
plt.show()


In [ ]:
# Cohort Comparison

from pyspark.sql.functions import when, datediff

# Count symptom visits per patient → pre_dx_visits
pre_dx_counts = (
    visits
    .filter(col("visit_type") == "SYMPTOM")
    .groupBy("subject_id")
    .agg(count("hadm_id").alias("pre_dx_visits"))
)

# Get BC diagnosis date per patient
bc_dx_dates = (
    visits
    .filter(col("visit_type") == "BC_FIRST_DX")
    .select(col("subject_id"), col("admit_day").alias("dx_date"))
)

# Get last symptom date per patient — used to compute days until dx
last_symptom = (
    visits
    .filter(col("visit_type") == "SYMPTOM")
    .groupBy("subject_id")
    .agg(spark_max("admit_day").alias("last_symptom_date"))
)

# Join everything and compute days between last symptom and diagnosis
patient_stats = (
    pre_dx_counts
    .join(bc_dx_dates, on="subject_id", how="inner")
    .join(last_symptom, on="subject_id", how="inner")
    .withColumn("days_before_dx", datediff(col("dx_date"), col("last_symptom_date")))
    .filter(col("days_before_dx") >= 0)
)

# Bucket patients by pre_dx_visits (matches sprint 4 shell script buckets)
patient_bucketed = (
    patient_stats
    .withColumn(
        "bucket",
        when(col("pre_dx_visits") == 1, "LOW")
        .when(col("pre_dx_visits") == 2, "MID")
        .otherwise("HIGH")
    )
)

# Group by bucket, compute distinct patients and averages
cohort_summary = (
    patient_bucketed
    .groupBy("bucket")
    .agg(
        count_distinct("subject_id").alias("distinct_patients"),
        mean("pre_dx_visits").alias("avg_pre_dx_visits"),
        mean("days_before_dx").alias("avg_days_before_dx")
    )
    .orderBy(
        when(col("bucket") == "LOW",  0)
        .when(col("bucket") == "MID",  1)
        .otherwise(2)
    )
)

print("=== cohort_summary ===")
cohort_summary.show(truncate=False)

# Bar chart — x: bucket, y: avg days before diagnosis
pdf = cohort_summary.toPandas()

fig_cohort_summary, ax = plt.subplots()
ax.bar(pdf["bucket"], pdf["avg_days_before_dx"])
ax.set_xlabel("Bucket (Pre-Diagnosis Visit Count)")
ax.set_ylabel("Avg Days Before Diagnosis")
ax.set_title("Patient Cohorts by Pre-Diagnosis Visit Frequency")
plt.show()

## Clean up
Stop spark session when done

In [ ]:
# Uncomment when you are completely done:

spark.stop()